# Titanic - Attempt 2
- **Autor:** Felipe Silva Loschi
- **Data:** abril/2026
- **Baseline (Attempt 1):** 0.77511 acurácia
- **Foco desta tentativa:** Feature Engineering adicionais, tentativa de novos modelos para verificar diferentes acurácias

> Como esta é uma continuação da tentativa anterior, insights, justificativas de escalonamento
> e Feature Engineering já documentados não serão repetidos — apenas as novidades serão detalhadas.

Importando as bibliotecas

In [25]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

Importando os dados

In [26]:
df = pd.read_csv('../../data/train.csv') #Estou rodando no VsCode, a importação pode variar de acordo com sua IDE
df

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


## Feature Engineering

In [27]:
df['Title'] = df['Name'].str.extract(r',\s*([^\.]+)\.') #Esse é o Regex que extrai isso em Python
df['Title'].value_counts()

Title
Mr              517
Miss            182
Mrs             125
Master           40
Dr                7
Rev               6
Col               2
Mlle              2
Major             2
Ms                1
Mme               1
Don               1
Lady              1
Sir               1
Capt              1
the Countess      1
Jonkheer          1
Name: count, dtype: int64

In [28]:
df['Title'] = df['Title'].replace({
    'Mlle':'Miss',
    'Mme':'Mrs',
    'Col':'Rare',
    'Major':'Rare',
    'Capt':'Rare',
    'Rev':'Rare',
    'Don': 'Rare',
    'Lady':'Rare',
    'Sir':'Rare',
    'the Countess':'Rare',
    'Jonkheer':'Rare'
})

Voltando a tentativa anterior, o título "Master" é usado para meninos jovens, então, quando for preencher valores nulos deles, posso colocar uma idade mais condizente que a mediana

Descobrindo quantos nulos serão tratados e as estatísticas antes

In [29]:
df[df['Title']=='Master']['Age'].isnull().sum()

np.int64(4)

In [30]:
df[df['Title']=='Master']['Age'].mean()

np.float64(4.574166666666667)

Tratando

In [31]:
df[df['Title']=='Master']['Age'].fillna(df[df['Title']=='Master']['Age'].median(), inplace=True)

C:\Users\lipe2\AppData\Local\Temp\ipykernel_3788\4083858519.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[df['Title']=='Master']['Age'].fillna(df[df['Title']=='Master']['Age'].median(), inplace=True)


Tá, vou ter que tratar de outra forma

In [32]:
df[(df['Title']=='Master') & (df['Age'].isna())]['Age'] = df[df['Title']=='Master']['Age'].median()

C:\Users\lipe2\AppData\Local\Temp\ipykernel_3788\3247680587.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[(df['Title']=='Master') & (df['Age'].isna())]['Age'] = df[df['Title']=='Master']['Age'].median()


Quase

In [33]:
df.loc[(df['Title']=='Master') & (df['Age'].isna()), 'Age'] = df[df['Title']=='Master']['Age'].median()

Agora sim, tive que usar '.loc[]' para acessar o dataframe original

In [34]:
df[df['Title']=='Master']['Age'].isnull().sum()

np.int64(0)

In [35]:
df[df['Title']=='Master']['Age'].mean()

np.float64(4.46675)

A média abaixou um pouco, o que é esperado, dado o valor que foi colocado onde era nulo ser 3.5, mas a Feature Engineering para esse caso agora foi mais bem feita.

Vou tratar o restante dos nulos da mesma forma que anteriormente

In [36]:
df["Age"].fillna(df["Age"].median(), inplace=True)
df["Cabin"].fillna("Unknown", inplace=True)
df["Embarked"].fillna(df["Embarked"].mode()[0], inplace=True)

C:\Users\lipe2\AppData\Local\Temp\ipykernel_3788\1325268493.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["Age"].fillna(df["Age"].median(), inplace=True)
C:\Users\lipe2\AppData\Local\Temp\ipykernel_3788\1325268493.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For exa

In [37]:
df.isna().sum()

PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Cabin          0
Embarked       0
Title          0
dtype: int64

Tudo foi tratado.

Continuando com as Feature Engineering, vou agora concatenar as colunas SibSp e Parch em uma única coluna. Dado que ambos falam sobre o tamanho da familia de uma pessoa, vou juntá-los em uma coluna chamada FamilySize e somar 1 nessa coluna, dado que SibSp e Parch me dão a informação apenas de quantos familiares a pessoa possuí, somando 1, ela mesma, eu tenho o tamanho da família.

In [38]:
df['FamilySize'] = df['Parch'] + df['SibSp'] + 1

## Treinando o Modelo
- Modelo: Em uma primeira tentativa, vou continuar com o Random Forest, mas também vou implementar XGBoost e o SVM dessa vez e vou tentar implementar uma ferramenta nova para mim, o StackingClassifier.

### Escalonamento
Primeiro, escalonamento. Colunas que vou usar:
- PassengerId: Não
- Name: Não
- Ticket: Não
- Cabin: Não
- Pclass: Sim
- Sex: Sim
- Age: Sim
- SibSp: Não(Informação já está em FamilySize)
- Parch: Não(Informação já está em FamilySize)
- FamilySize: Sim
- Fare: Sim
- Embarked: Sim
- Title: Sim

In [57]:
X = df.drop(columns=['Survived', 'SibSp', 'Parch', 'PassengerId',
                     'Name', 'Cabin', 'Ticket'], axis=1)
y = df['Survived']

In [58]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [59]:
X_train.columns

Index(['Pclass', 'Sex', 'Age', 'Fare', 'Embarked', 'Title', 'FamilySize'], dtype='object')

### Escalonamento para numéricos
- Padronização na variável nova e nas antigas
- Pclass continua sem escalonamento

### Escalonamento para não-numéricos
- Continua como antes

In [60]:
numeric_features = ['Age', 'FamilySize', 'Fare']
categorical_features = ['Sex', 'Embarked', 'Title']

In [61]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(drop='first'), categorical_features)
    ],
    remainder='passthrough' #Pra ele pegar o Pclass também
)

### Treinando o modelo

In [82]:
pipeline = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("model", RandomForestClassifier(max_depth = 9))
])

In [83]:
pipeline.fit(X_train, y_train)

c:\Users\lipe2\anaconda3\Lib\site-packages\sklearn\compose\_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('preprocessing',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num', StandardScaler(),
                                                  ['Age', 'FamilySize',
                                                   'Fare']),
                                                 ('cat',
                                                  OneHotEncoder(drop='first'),
                                                  ['Sex', 'Embarked',
                                                   'Title'])])),
                ('model', RandomForestClassifier(max_depth=9))])

In [84]:
y_train_predict = pipeline.predict(X_train)
print(f"Acurácia do modelo: {accuracy_score(y_train, y_train_predict):.2f}")

Acurácia do modelo: 0.92


In [85]:
y_test_predict = pipeline.predict(X_test)
print(f"Acurácia do modelo: {accuracy_score(y_test, y_test_predict):.2f}")

Acurácia do modelo: 0.84


Valor dos parâmetro do RandomForest foi decidido por tentativas.
Para max_depth:
- Para 4: Treino = 85%, Teste = 82%
- Para 5: Treino = 86%, Teste = 82%
- Para 6: Treino = 87%, Teste = 83%
- Para 7: Treino = 89%, Teste = 83%
- Para 8: Treino = 90%, Teste = 83%
- Para 9: Treino = 92%, Teste = 84%

- Vou aprender como ajustar os parâmetros com alguma função